# 03 The Association Between Exposure and Disease — Reference Solutions

Complete solutions to the exercises for the Legionnaires' disease cluster at Songbai Nursing Home.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact
from epi_learning.metrics import risk_ratio, odds_ratio

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

## Question 1: A Complete Analysis of COPD × Infection

In [ ]:
ct_copd = pd.crosstab(
    df["comorbidity_copd"], df["infected"],
    margins=True, margins_name="Total",
)
ct_copd.index = ["No COPD", "Has COPD", "Total"]
ct_copd.columns = ["Not infected", "Infected", "Total"]
print(ct_copd)

a = int(ct_copd.loc["Has COPD", "Infected"])
b = int(ct_copd.loc["Has COPD", "Not infected"])
c = int(ct_copd.loc["No COPD", "Infected"])
d = int(ct_copd.loc["No COPD", "Not infected"])

rr = risk_ratio(a, a + b, c, c + d)
or_val = odds_ratio(a, b, c, d)

ln_rr = np.log(rr)
se_rr = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
ci_rr_lo = np.exp(ln_rr - 1.96 * se_rr)
ci_rr_hi = np.exp(ln_rr + 1.96 * se_rr)

ln_or = np.log(or_val)
se_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_or_lo = np.exp(ln_or - 1.96 * se_or)
ci_or_hi = np.exp(ln_or + 1.96 * se_or)

chi2, p, _, _ = chi2_contingency([[a, b], [c, d]])

print(f"\nCOPD -> infection")
print(f"  RR = {rr:.3f} (95% CI: {ci_rr_lo:.3f} – {ci_rr_hi:.3f})")
print(f"  OR = {or_val:.3f} (95% CI: {ci_or_lo:.3f} – {ci_or_hi:.3f})")
print(f"  chi-square = {chi2:.3f}, p-value = {p:.4f}")
print(f"  RR vs OR difference: {abs(or_val - rr):.3f} (when attack rate is high, OR > RR)")

if ci_rr_lo > 1:
    print("  -> COPD is a statistically significant risk factor for infection")
else:
    print("  -> COPD has no statistically significant association with infection (CI contains 1)")

## Question 2: Ranking Comorbidities by RR / OR

In [ ]:
comorbidities = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd",
    "immunosuppressed",
]

results = []
for factor in comorbidities:
    ct = pd.crosstab(df[factor], df["infected"])
    a_i = int(ct.loc[1, 1])
    b_i = int(ct.loc[1, 0])
    c_i = int(ct.loc[0, 1])
    d_i = int(ct.loc[0, 0])
    rr_i = risk_ratio(a_i, a_i + b_i, c_i, c_i + d_i)
    or_i = odds_ratio(a_i, b_i, c_i, d_i)
    chi2_i, p_i, _, _ = chi2_contingency([[a_i, b_i], [c_i, d_i]])
    ln_rr_i = np.log(rr_i)
    se_i = np.sqrt(1/a_i - 1/(a_i+b_i) + 1/c_i - 1/(c_i+d_i))
    ci_lo = np.exp(ln_rr_i - 1.96 * se_i)
    ci_hi = np.exp(ln_rr_i + 1.96 * se_i)
    results.append({
        "Comorbidity": factor.replace("comorbidity_", "").upper()
                if "comorbidity_" in factor else factor.upper(),
        "RR": round(rr_i, 3),
        "95% CI": f"{ci_lo:.3f}–{ci_hi:.3f}",
        "OR": round(or_i, 3),
        "p-value": round(p_i, 4),
        "Significant": "*" if ci_lo > 1 else "",
        "RR-OR diff": round(abs(or_i - rr_i), 3),
    })

rr_df = pd.DataFrame(results).sort_values("RR", ascending=False)
print("=== Comorbidity RR / OR Ranking ===")
print(rr_df.to_string(index=False))
print("\n-> The comorbidity with the highest RR also has the highest attack rate, so OR deviates from RR the most")

## Question 3: Sex Difference Analysis

In [ ]:
df["is_male"] = (df["sex"] == "M").astype(int)

ct_sex = pd.crosstab(df["is_male"], df["infected"])
a_s = int(ct_sex.loc[1, 1])
b_s = int(ct_sex.loc[1, 0])
c_s = int(ct_sex.loc[0, 1])
d_s = int(ct_sex.loc[0, 0])

rr_sex = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
or_sex = odds_ratio(a_s, b_s, c_s, d_s)
chi2_s, p_s, _, _ = chi2_contingency([[a_s, b_s], [c_s, d_s]])

ln_rr_s = np.log(rr_sex)
se_s = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
ci_lo_s = np.exp(ln_rr_s - 1.96 * se_s)
ci_hi_s = np.exp(ln_rr_s + 1.96 * se_s)

print(f"Male vs Female -> infection")
print(f"  RR = {rr_sex:.3f} (95% CI: {ci_lo_s:.3f} – {ci_hi_s:.3f})")
print(f"  OR = {or_sex:.3f}")
print(f"  p-value = {p_s:.4f}")

print(f"\n=== CFR by Sex ===")
for sex_label in ["M", "F"]:
    infected_sex = df[(df["sex"] == sex_label) & (df["infected"] == 1)]
    deaths_sex = (infected_sex["outcome"] == "dead").sum()
    n_infected = len(infected_sex)
    cfr = deaths_sex / n_infected if n_infected > 0 else 0
    print(f"  {sex_label}: CFR = {cfr:.1%} ({deaths_sex}/{n_infected})")

print("\nInterpretation:")
print("- Infection risk (RR): measures susceptibility -- whether sex affects the probability of infection")
print("- Case fatality rate (CFR): measures prognosis -- whether sex affects survival after infection")
print("- These are two different questions and need to be analyzed separately")

## Question 4: Cohort Study vs. Case-Control Study

In [ ]:
# This is a case-control study
# The researcher "found cases first, then selected controls," rather than following an entire population
# The denominator is artificially set (30:60) and doesn't reflect the true disease incidence
# So (24/39) / (6/51) is not a true "risk ratio"

a, b, c, d = 24, 15, 6, 45
or_val = odds_ratio(a, b, c, d)

ln_or = np.log(or_val)
se_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_lo = np.exp(ln_or - 1.96 * se_or)
ci_hi = np.exp(ln_or + 1.96 * se_or)

oddsr_f, p_fisher = fisher_exact([[a, b], [c, d]])

print("=== Case-Control Study: Salad x Food Poisoning ===")
print(f"OR = {or_val:.3f} (95% CI: {ci_lo:.3f} – {ci_hi:.3f})")
print(f"Fisher's exact test: p = {p_fisher:.6f}")
print(f"\nInterpretation: salad eaters' odds of infection are {or_val:.1f}x those of non-eaters")
print("CI does not contain 1 and the p-value is tiny -> salad has a statistically significant association with food poisoning")
print("\nIf we could obtain data on all 500 people -> it becomes a retrospective cohort study -> we could compute RR")

## Question 5 (Challenge): Fisher vs. Chi-Square for Small Samples

In [ ]:
subset = df[(df["floor"] == 3) & (df["wing"] == "B")].copy()
print(f"Floor 3 Wing B residents: {len(subset)} people")
print(f"Number infected: {subset['infected'].sum()} people")

ct_sub = pd.crosstab(subset["shower_use"], subset["infected"])
print(f"\n2x2 table:")
print(ct_sub)

a_sub = int(ct_sub.loc[1, 1]) if 1 in ct_sub.index and 1 in ct_sub.columns else 0
b_sub = int(ct_sub.loc[1, 0]) if 1 in ct_sub.index and 0 in ct_sub.columns else 0
c_sub = int(ct_sub.loc[0, 1]) if 0 in ct_sub.index and 1 in ct_sub.columns else 0
d_sub = int(ct_sub.loc[0, 0]) if 0 in ct_sub.index and 0 in ct_sub.columns else 0

chi2_sub, p_chi2, dof, expected = chi2_contingency([[a_sub, b_sub], [c_sub, d_sub]])
print(f"\nExpected-value table:")
print(pd.DataFrame(expected.round(2),
                   index=["No shower", "Shower"],
                   columns=["Not infected", "Infected"]))

min_exp = expected.min()
print(f"\nMinimum expected value = {min_exp:.2f}", end="")
if min_exp < 5:
    print(" -> < 5, the chi-square approximation may be inaccurate!")
else:
    print(" -> >= 5, the chi-square test applies")

oddsr_f, p_fisher = fisher_exact([[a_sub, b_sub], [c_sub, d_sub]])
print(f"\nChi-square test: χ² = {chi2_sub:.3f}, p = {p_chi2:.4f}")
print(f"Fisher's exact test: p = {p_fisher:.4f}")
print(f"p-value difference: {abs(p_chi2 - p_fisher):.4f}")
print("\nConclusion:")
print("- With small samples, Fisher's exact test is more reliable (it does not rely on a large-sample approximation)")
print("- The chi-square test may over- or under-estimate significance when expected values are < 5")
print("- In practice, if more than 20% of cells have expected values < 5, switch to Fisher")

### Interpretation

**Key points for Question 4:**
- A **case-control study** can't compute the RR, because the denominator is artificially set by the researcher and doesn't reflect the true disease incidence in the population
- The **OR** is the correct effect measure for a case-control study
- If you could obtain data on everyone, it would become a cohort study and you could compute the RR

**Key points for Question 5:**
- For small samples, the chi-square test's χ² distribution approximation isn't accurate enough
- **Fisher's exact test** directly computes the exact probability of observing the current or more extreme results under H₀
- Rule of thumb: when any cell has an expected value < 5, use Fisher first